# 01 — Generate the Logistics Q&A Dataset

Generates ~6,000 question/answer pairs using the Anthropic API.

**Before you run this notebook:**
1. Get an Anthropic API key from https://console.anthropic.com/
2. **Set a monthly spending cap** on the Anthropic console (Settings → Billing → Usage limits). Even $50 is enough; this run costs ~$3-8.
3. In Colab: click the 🔑 key icon in the left sidebar → **Add new secret** → Name: `ANTHROPIC_API_KEY`, Value: your key, toggle Notebook access ON.

**Runtime:** ~6 hours on free Colab. Either GPU or CPU runtime works (dataset gen doesn't use GPU — saves your GPU quota for notebook 02).

**Recovery:** The backup loop cell saves progress to Drive every 5 minutes. If Colab disconnects, just restart cell 11 — it resumes from the last backup automatically.

## Setup

In [ ]:
# Clone the repo and install deps
!git clone https://github.com/masonsau0/logistics-qa-lora.git
%cd logistics-qa-lora
!pip install -q anthropic datasets python-dotenv

In [ ]:
# Load API key from Colab secrets (🔑 icon in left sidebar)
import os

from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
assert os.environ["ANTHROPIC_API_KEY"].startswith("sk-ant-"), "Key looks malformed"
print("API key loaded")

## Mount Google Drive and restore any prior backup

Mounts Drive so the backup loop (later) and final save can write to it. The restore cell pulls in any `raw_generated.jsonl` from a previous run so the full-run cell can pick up where it left off.

First-time run with no prior backup: the restore cell is a no-op.

In [ ]:
# Mount Drive — watch for the Google auth popup and click through it
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Restore from Drive backup if local doesn't have data yet
import os

local = "data/raw_generated.jsonl"
backup = "/content/drive/MyDrive/logistics-qa-lora/data/raw_generated.jsonl"

if os.path.exists(local):
    n_local = sum(1 for _ in open(local))
    print(f"Local already has {n_local} examples — skipping restore.")
elif os.path.exists(backup):
    !cp {backup} {local}
    n = sum(1 for _ in open(local))
    print(f"Restored {n} examples from Drive backup")
else:
    print("No local data and no Drive backup — the full run will start from 0")

## Smoke test (optional, ~$0.10)

Generates 1 batch (~48 examples) per category to verify the pipeline works before spending real money on the full run.

**Skip this if you've already validated the pipeline.** Run only for first-time setup.

In [ ]:
# Generates ~50 examples (1 batch per category) — should finish in ~1 minute and cost < $0.10
!python -m data.prepare_dataset --smoke --batch-size 8

In [ ]:
# Inspect a few generated examples
import json

with open("data/raw_generated.jsonl") as f:
    for line in list(f)[:3]:
        rec = json.loads(line)
        print(f"[{rec['category']}]")
        print(f"Q: {rec['question']}")
        print(f"A: {rec['answer'][:200]}...")
        print(f"Key facts: {rec['key_facts']}")
        print()

## Full run (~6 hours)

Generates the full 6K-example dataset and writes train/val/test splits.

Resumable — if Colab disconnects, just restart this cell. The backup thread (inside this same cell) saves to Drive every 5 minutes so disconnects lose at most 5 minutes of progress.

**Important — Colab quirk:** This cell runs both the script AND a backup thread together because Colab's kernel runs cells sequentially. A separate backup cell would just queue and never run until the script finished (defeating its purpose). The threading inside ONE cell sidesteps that.

You'll see two kinds of output interleaved:
- Script batch logs: `[freight_calculations] +8 (total new this run: 8)`
- Backup logs every 5 min: `[backup] N examples → Drive at HH:MM:SS`

Walk away for ~6 hours. Click in the Colab tab every ~60 minutes to prevent idle disconnect.

In [ ]:
import os
import shutil
import subprocess
import threading
import time

# Source = local file the script writes; Dest = Drive copy that survives VM resets
src = "/content/logistics-qa-lora/data/raw_generated.jsonl"
dst = "/content/drive/MyDrive/logistics-qa-lora/data/raw_generated.jsonl"
os.makedirs(os.path.dirname(dst), exist_ok=True)


def backup_loop():
    while True:
        if os.path.exists(src):
            try:
                shutil.copy2(src, dst)
                n = sum(1 for _ in open(src))
                print(f"[backup] {n} examples → Drive at {time.strftime('%H:%M:%S')}", flush=True)
            except Exception as e:
                print(f"[backup error] {e}", flush=True)
        time.sleep(300)


# Start backup thread — runs in parallel with the subprocess below
threading.Thread(target=backup_loop, daemon=True).start()
print("Backup thread started — running in parallel with the generator below", flush=True)

# Run the generator script — blocks this cell, but the backup thread keeps going
subprocess.run(
    ["python", "-m", "data.prepare_dataset", "--target", "6000", "--batch-size", "8", "--split"],
    cwd="/content/logistics-qa-lora",
    check=False,
)
print("Script finished.", flush=True)

## After the full run finishes — verify split sizes

Run this only AFTER the full-run cell above completes. The split files don't exist until then.

In [ ]:
# Verify split sizes
for split in ["train", "val", "test"]:
    n = sum(1 for _ in open(f"data/{split}.jsonl"))
    print(f"{split}: {n}")

## Save final splits to Google Drive

Copies the train/val/test splits to Drive so they survive between Colab sessions. Notebook 02 will restore them from here.

In [ ]:
!mkdir -p /content/drive/MyDrive/logistics-qa-lora/data
!cp data/*.jsonl /content/drive/MyDrive/logistics-qa-lora/data/
!ls -lh /content/drive/MyDrive/logistics-qa-lora/data/

### Next step

Open `02_train_lora.ipynb`. A fresh Colab session is fine — Drive persists across sessions, so the splits will be there.